Setup and installs

In [1]:
# Step 1: Install libraries
!pip install transformers datasets pandas pyarrow scikit-learn torch -q
print("Libraries installed.")

Libraries installed.


Config

Load data and prepare ground truth spans

In [2]:
# Step 2: Load data, Parse Numpy Arrays in trigger_words, and Verify

import pandas as pd
import ast
import numpy as np
from datasets import Dataset # Import Dataset here

# --- Revised parsing function to handle numpy arrays ---
def parse_trigger_words_revised(row):
    # Handle outer numpy array case (most common based on inspection)
    if isinstance(row, np.ndarray):
        cleaned = []
        # Iterate through items stored within the numpy object array
        for item in row:
            # Check if item looks like a coordinate pair (list, tuple, or another ndarray)
            if isinstance(item, (np.ndarray, list, tuple)) and len(item) == 2:
                 try:
                      # Convert elements to int
                      start = int(item[0])
                      end = int(item[1])
                      # Optional sanity check:
                      if start >= 0 and end >= start:
                           cleaned.append([start, end]) # Standardize to list
                 except (ValueError, TypeError):
                      continue # Skip if elements aren't convertible to int
        return cleaned

    # Handle cases where it might already be a standard list/tuple (less likely now)
    elif isinstance(row, (list, tuple)):
        cleaned = []
        for item in row:
             if isinstance(item, (list, tuple)) and len(item) == 2:
                  try:
                       start = int(item[0])
                       end = int(item[1])
                       if start >= 0 and end >= start:
                           cleaned.append([start, end])
                  except (ValueError, TypeError): continue
        return cleaned

    # Handle None/NaN (as before)
    elif row is None or pd.isna(row):
        return []

    # Handle string case (less likely based on inspection, but keep for robustness)
    elif isinstance(row, str):
        row_str = row.strip()
        if row_str == "None" or not row_str: return []
        try:
             parsed = ast.literal_eval(row_str) # Only handles basic list strings
             if isinstance(parsed, list):
                  cleaned = []
                  for item in parsed:
                       if isinstance(item, (list, tuple)) and len(item) == 2:
                            try:
                                 start = int(item[0])
                                 end = int(item[1])
                                 if start >= 0 and end >= start:
                                     cleaned.append([start, end])
                            except (ValueError, TypeError): continue
                  return cleaned
             else: return []
        except:
             # Failed simple parsing, likely the complex "[array...]" string or something else
             return [] # Return empty if string parsing fails

    # Handle any other unexpected type
    else:
        return []
# --- End of revised parsing function ---

# Load the parquet file
file_path = 'train.parquet' # Make sure this file is uploaded
try:
    df_span = pd.read_parquet(file_path)
    print(f"Loaded data with columns: {df_span.columns.to_list()}")
    print(f"Total rows loaded: {len(df_span)}")

    # --- Apply the REVISED parsing function ---
    df_span["trigger_spans"] = df_span["trigger_words"].apply(parse_trigger_words_revised)
    print("Applied REVISED parser to 'trigger_words' -> 'trigger_spans'.")

    # Rename content column
    df_span.rename(columns={"content": "text"}, inplace=True)
    print("Renamed 'content' to 'text'.")

    # --- Verification Block (Check if parsing worked) ---
    print("\nVerifying trigger_spans in DataFrame df_span...")
    if 'trigger_spans' in df_span.columns:
        num_positive_in_df = df_span[df_span['trigger_spans'].apply(lambda x: len(x) > 0)].shape[0]
        total_rows_in_df = df_span.shape[0]
        print(f"Total rows in DataFrame: {total_rows_in_df}")
        print(f"Number of rows with non-empty trigger_spans: {num_positive_in_df}")
        if num_positive_in_df == 0:
            print("Error: DataFrame STILL contains NO examples with parsed manipulation spans!")
            print("Check the train.parquet file contents and the revised parser logic.")
        else:
            print("Success: DataFrame contains examples with parsed manipulation spans.")
    else:
        print("Error: 'trigger_spans' column was not created.")
    # --- End of verification block ---

    # --- Create Hugging Face Dataset (only if verification passed) ---
    if 'trigger_spans' in df_span.columns and num_positive_in_df > 0:
        # Select columns needed
        df_subset = df_span[["text", "trigger_spans"]].copy()
        # Convert to Hugging Face Dataset
        raw_dataset = Dataset.from_pandas(df_subset)
        print("\nCreated raw Hugging Face Dataset:")
        print(raw_dataset)
    elif 'trigger_spans' in df_span.columns and num_positive_in_df == 0:
        print("\nSkipping Hugging Face Dataset creation due to lack of positive examples.")
        raw_dataset = None # Ensure variable exists but is None
    else:
        print("\nSkipping Hugging Face Dataset creation due to missing column.")
        raw_dataset = None


except FileNotFoundError:
    print(f"Error: {file_path} not found. Please upload the training data.")
    raw_dataset = None
except Exception as e:
    print(f"An error occurred during data loading/parsing: {e}")
    raw_dataset = None

Loaded data with columns: ['id', 'content', 'lang', 'manipulative', 'techniques', 'trigger_words']
Total rows loaded: 3822
Applied REVISED parser to 'trigger_words' -> 'trigger_spans'.
Renamed 'content' to 'text'.

Verifying trigger_spans in DataFrame df_span...
Total rows in DataFrame: 3822
Number of rows with non-empty trigger_spans: 2589
Success: DataFrame contains examples with parsed manipulation spans.

Created raw Hugging Face Dataset:
Dataset({
    features: ['text', 'trigger_spans'],
    num_rows: 3822
})


Load fine-tuned model and tokenizer

In [3]:
# Step 3: Load BASE model and tokenizer

from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch # Ensure torch is imported

model_path = "benjamin/roberta-large-wechsel-ukrainian"
try:
    print(f"Loading tokenizer for: {model_path}")
    # Ensure use_fast=True if available and desired
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

    print(f"Loading BASE model for token classification: {model_path}")
    # Load the model configured for BINARY classification (num_labels=2)
    model = AutoModelForTokenClassification.from_pretrained(
        model_path,
        num_labels=2, # For binary classification (0 or 1)
        # problem_type="token_classification", # Usually inferred
    )
    print("\nBase model and tokenizer loaded successfully.")

except Exception as e:
    print(f"An error occurred loading the model/tokenizer: {e}")
    model = None
    tokenizer = None

Loading tokenizer for: benjamin/roberta-large-wechsel-ukrainian


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading BASE model for token classification: benjamin/roberta-large-wechsel-ukrainian


Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at benjamin/roberta-large-wechsel-ukrainian and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Base model and tokenizer loaded successfully.


Split dataset

In [4]:
# Step 4: Split Dataset

if 'raw_dataset' in locals() and raw_dataset is not None:
    # Split the raw dataset (contains 'text' and 'trigger_spans')
    # Use test_size=0.2 for 80/20 split as in baseline
    split_dataset = raw_dataset.train_test_split(test_size=0.2, seed=42)
    train_split = split_dataset["train"]
    eval_split = split_dataset["test"] # Keep this for later evaluation of spans

    print("Split raw dataset into train and evaluation sets:")
    print(f"Training examples: {len(train_split)}")
    print(f"Evaluation examples: {len(eval_split)}")

    # --- Verify the new splits ---
    num_train_positive_split = sum(1 for example in train_split if len(example['trigger_spans']) > 0)
    num_eval_positive_split = sum(1 for example in eval_split if len(example['trigger_spans']) > 0)
    print(f"Train split positive examples: {num_train_positive_split}")
    print(f"Eval split positive examples: {num_eval_positive_split}")
    if num_eval_positive_split == 0:
         print("WARNING: Eval split still has 0 positive examples after simple split!")
    # --- End verification ---

else:
    print("Error: raw_dataset not loaded or created in Step 2.")

Split raw dataset into train and evaluation sets:
Training examples: 3057
Evaluation examples: 765
Train split positive examples: 2075
Eval split positive examples: 514


In [5]:
# New Cell: Verify Data Split (Run AFTER Step 4)

if 'train_split' in locals() and 'eval_split' in locals():
    num_train_positive = sum(1 for example in train_split if len(example['trigger_spans']) > 0)
    num_eval_positive = sum(1 for example in eval_split if len(example['trigger_spans']) > 0)

    print(f"Train split: {len(train_split)} total examples, {num_train_positive} have manipulation spans.")
    print(f"Eval split: {len(eval_split)} total examples, {num_eval_positive} have manipulation spans.")

    if num_eval_positive == 0:
        print("\nError: Evaluation split has NO examples with manipulation spans!")
        print("This evaluation is not meaningful. Consider using stratified splitting.")
    else:
        print("\nEvaluation split contains positive examples.")
else:
    print("Error: train_split or eval_split not found. Ensure Step 4 ran.")

Train split: 3057 total examples, 2075 have manipulation spans.
Eval split: 765 total examples, 514 have manipulation spans.

Evaluation split contains positive examples.


Tokenize and align binary labels

In [6]:
# Step 5: Tokenize and Align Binary Labels

# --- Copy the function definition from BaseLine_1.py ---
def tokenize_and_align_spans(example):
    # Tokenize the text and return offsets
    # Make sure 'tokenizer' is loaded from Step 3
    tokenized_inputs = tokenizer(
        example["text"],
        truncation=True,
        # Using max_length from the baseline script
        padding="max_length", # Pad to max_length for simplicity here
        max_length=512,
        return_offsets_mapping=True,
    )

    # Pop the offset mapping
    offset_mapping = tokenized_inputs.pop("offset_mapping")

    # Initialize token-level labels: 0 for non-manipulative
    labels = [0] * len(offset_mapping)

    # Get the trigger spans for this example
    spans = example["trigger_spans"] # Assumes 'trigger_spans' column exists

    # For each token, check if its character offsets overlap with any trigger span.
    for i, (token_start, token_end) in enumerate(offset_mapping):
        # Label special tokens/padding as -100
        if token_start == token_end:
            labels[i] = -100
            continue

        # Check each span for overlap
        for span in spans:
            span_start, span_end = span
            # Overlap condition: token ends after span starts AND token starts before span ends
            if token_end > span_start and token_start < span_end:
                labels[i] = 1 # Mark as 1 (manipulative)
                break  # No need to check other spans for this token

    tokenized_inputs["labels"] = labels
    return tokenized_inputs
# --- End of function definition ---

# Apply the function to the training and evaluation splits
# Make sure train_split and eval_split exist from Step 4
if 'train_split' in locals() and 'eval_split' in locals() and tokenizer is not None:
    print("Tokenizing train_split...")
    tokenized_train_dataset = train_split.map(
        tokenize_and_align_spans,
        batched=False # Process one by one as in baseline script
        # Consider removing text/trigger_spans columns if no longer needed for training
        # remove_columns=train_split.column_names
    )

    print("Tokenizing eval_split...")
    tokenized_eval_dataset = eval_split.map(
        tokenize_and_align_spans,
        batched=False
        # remove_columns=eval_split.column_names
    )

    print("\nProcessed datasets created:")
    print("Tokenized Training Dataset:", tokenized_train_dataset)
    print("Tokenized Evaluation Dataset:", tokenized_eval_dataset)

    # Optional: Inspect one example
    # print("\nProcessed Training Example 0:")
    # print(tokenized_train_dataset[0])
    # print("\nDecoded Tokens and Labels (Binary):")
    # example = tokenized_train_dataset[0]
    # tokens = tokenizer.convert_ids_to_tokens(example['input_ids'])
    # label_names = [str(l) if l != -100 else 'IGN' for l in example['labels']]
    # for token, label in zip(tokens[:50], label_names[:50]):
    #       print(f"{token:<15} {label}")

else:
    print("Error: Ensure raw_dataset was split in Step 4 and tokenizer loaded in Step 3.")

Tokenizing train_split...


Map:   0%|          | 0/3057 [00:00<?, ? examples/s]

Tokenizing eval_split...


Map:   0%|          | 0/765 [00:00<?, ? examples/s]


Processed datasets created:
Tokenized Training Dataset: Dataset({
    features: ['text', 'trigger_spans', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3057
})
Tokenized Evaluation Dataset: Dataset({
    features: ['text', 'trigger_spans', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 765
})


Set Dataset Format

In [7]:
# Step 6: Set Dataset Format

if 'tokenized_train_dataset' in locals() and 'tokenized_eval_dataset' in locals():
    # Set format for PyTorch
    tokenized_train_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"] # Specify columns needed for model
    )
    tokenized_eval_dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )
    print("Dataset format set to PyTorch.")
    # You can optionally check the format
    # print(tokenized_train_dataset)
else:
    print("Error: Tokenized datasets not found. Please ensure Step 5 ran successfully.")

Dataset format set to PyTorch.


Define Compute Metrics

In [8]:
# Step 7: Define Compute Metrics Function (with Debugging)

import numpy as np
from sklearn.metrics import f1_score, classification_report # Import classification_report

def compute_token_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Flatten predictions and labels, ignoring -100
    preds_flat = []
    labels_flat = []
    for p_seq, l_seq in zip(preds, labels):
        for pred, lab in zip(p_seq, l_seq):
            if lab != -100: # Only consider non-ignored tokens
                preds_flat.append(pred)
                labels_flat.append(lab)

    # --- Add Debugging ---
    print(f"\n--- Inside compute_token_metrics ---")
    print(f"Number of non-ignored tokens: {len(labels_flat)}")
    unique_labels, counts_labels = np.unique(labels_flat, return_counts=True)
    unique_preds, counts_preds = np.unique(preds_flat, return_counts=True)
    print(f"True label counts: {dict(zip(unique_labels, counts_labels))}")
    print(f"Predicted label counts: {dict(zip(unique_preds, counts_preds))}")
    # --- End Debugging ---

    # Calculate overall macro F1
    f1_macro = f1_score(labels_flat, preds_flat, average="macro", zero_division=0)

    # Calculate per-class F1 (and other metrics)
    report = classification_report(labels_flat, preds_flat, output_dict=True, zero_division=0)
    f1_class_0 = report.get('0', {}).get('f1-score', 0)
    f1_class_1 = report.get('1', {}).get('f1-score', 0)
    # Also get precision and recall for class 1 (Manipulative)
    precision_class_1 = report.get('1', {}).get('precision', 0)
    recall_class_1 = report.get('1', {}).get('recall', 0)

    print(f"Class 0 F1: {f1_class_0:.4f}, Class 1 F1: {f1_class_1:.4f}")
    print(f"Class 1 Precision: {precision_class_1:.4f}, Class 1 Recall: {recall_class_1:.4f}")
    print(f"Overall Macro F1: {f1_macro:.4f}")
    print(f"--- End compute_token_metrics ---")


    # Return metrics
    # Returning class 1 metrics might be more informative than just macro F1
    return {
        "token_macro_f1": f1_macro,
        "token_f1_class0": f1_class_0,
        "token_f1_class1": f1_class_1,
        "token_precision_class1": precision_class_1,
        "token_recall_class1": recall_class_1,
         }

print("compute_token_metrics function defined (with debugging and per-class metrics).")

compute_token_metrics function defined (with debugging and per-class metrics).


Define Training args

In [9]:
# Step 8: Define Training Arguments

from transformers import TrainingArguments

# Define output directory for this training run
output_dir = "./token+span_approach_testing"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",        # Use corrected name: eval_strategy
    eval_steps=100,               # Evaluate every 100 steps
    save_steps=100,               # Save a checkpoint every 100 steps
    num_train_epochs=4,           # Use 8 epochs as in BaseLine_1.py
    per_device_train_batch_size=16,# Use 32 as in BaseLine_1.py
    per_device_eval_batch_size=16,# Use 32 as in BaseLine_1.py
    learning_rate=2e-5,           # Use 1e-5 as in BaseLine_1.py
    weight_decay=0.01,
    fp16=True,                    # Use mixed precision (make sure GPU is T4 or newer)
    logging_steps=50,
    save_total_limit=2,           # Keep only the two most recent checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="token_f1_class1", # <<< Monitor F1 for class 1
    greater_is_better=True,
    report_to="none"              # Disable external reporting
)

print("Training arguments defined.")

Training arguments defined.


Initialize Trainer

In [10]:
# Step 9: Initialize the Trainer

from transformers import Trainer, DataCollatorForTokenClassification # Ensure imports

# Define a simple data collator (as Trainer needs one, though not strictly necessary if dataset pads)
# The baseline script didn't explicitly define one, but Trainer usually requires it.
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Ensure all components are defined from previous steps:
# model: loaded in Step 3
# training_args: defined in Step 8
# tokenized_train_dataset: created in Step 5
# tokenized_eval_dataset: created in Step 5
# compute_token_metrics: defined in Step 7

if 'model' in locals() and \
   'training_args' in locals() and \
   'tokenized_train_dataset' in locals() and \
   'tokenized_eval_dataset' in locals() and \
   'tokenizer' in locals() and \
   'compute_token_metrics' in locals():

    trainer = Trainer(
        model=model,                        # The RoBERTa model loaded in Step 3
        args=training_args,                 # Args defined in Step 8
        train_dataset=tokenized_train_dataset, # Tokenized train data from Step 5
        eval_dataset=tokenized_eval_dataset,  # Tokenized eval data from Step 5
        tokenizer=tokenizer,                # Tokenizer loaded in Step 3
        data_collator=data_collator,        # Simple data collator
        compute_metrics=compute_token_metrics # Metric function from Step 7
    )
    print("Trainer initialized successfully.")

else:
    print("Error: One or more required components (model, args, datasets, tokenizer, compute_metrics) are missing.")
    print("Please ensure Steps 3, 5, 7, and 8 completed successfully.")

<ipython-input-10-c29ace6bf0ac>:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Trainer initialized successfully.


Model Training

In [11]:
# Step 10: Train the Model

if 'trainer' in locals():
    print("Starting training (Binary Token Classification)...")
    train_result = trainer.train()
    print("Training finished!")

    # ----- After Training -----

    # Optional: Log and save training metrics
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)

    # Evaluate the BEST model on the evaluation set
    print("\nEvaluating best model on the evaluation set...")
    eval_results = trainer.evaluate()
    print("Token classification evaluation results:", eval_results)
    trainer.log_metrics("eval", eval_results)
    trainer.save_metrics("eval", eval_results)


    # Save the final best model and tokenizer
    # (Trainer already loaded the best model based on 'token_macro_f1')
    save_directory = training_args.output_dir # Use the directory from training args
    print(f"\nSaving the best model to {save_directory}...")
    trainer.save_model(save_directory)
    # Save the tokenizer associated with the trainer/model
    if hasattr(trainer, 'tokenizer') and trainer.tokenizer is not None:
         trainer.tokenizer.save_pretrained(save_directory)
         print("Tokenizer also saved.")
    else:
         # If tokenizer wasn't passed to trainer init (it was), try saving global one
         try:
              tokenizer.save_pretrained(save_directory)
              print("Tokenizer also saved (using global variable).")
         except Exception as e:
              print(f"Could not save tokenizer automatically: {e}")

    print(f"Best model saved to {save_directory}")

else:
    print("Error: Trainer object not found. Please ensure Step 9 ran successfully.")

Starting training (Binary Token Classification)...


Step,Training Loss,Validation Loss,Token Macro F1,Token F1 Class0,Token F1 Class1,Token Precision Class1,Token Recall Class1
100,0.458800,0.410219,0.729551,0.869001,0.590101,0.573835,0.607317
200,0.407500,0.402296,0.704071,0.888920,0.519221,0.695867,0.414102
300,0.393400,0.404454,0.741022,0.868715,0.613329,0.572356,0.660621
400,0.368600,0.443899,0.700415,0.890205,0.510625,0.713356,0.397623
500,0.300500,0.391113,0.744969,0.883485,0.606452,0.625312,0.588697
600,0.276900,0.433267,0.736690,0.885123,0.588257,0.638552,0.545307
700,0.259300,0.421777,0.736706,0.885645,0.587766,0.641407,0.542405



--- Inside compute_token_metrics ---
Number of non-ignored tokens: 123010
True label counts: {np.int64(0): np.int64(94063), np.int64(1): np.int64(28947)}
Predicted label counts: {np.int64(0): np.int64(92374), np.int64(1): np.int64(30636)}
Class 0 F1: 0.8690, Class 1 F1: 0.5901
Class 1 Precision: 0.5738, Class 1 Recall: 0.6073
Overall Macro F1: 0.7296
--- End compute_token_metrics ---

--- Inside compute_token_metrics ---
Number of non-ignored tokens: 123010
True label counts: {np.int64(0): np.int64(94063), np.int64(1): np.int64(28947)}
Predicted label counts: {np.int64(0): np.int64(105784), np.int64(1): np.int64(17226)}
Class 0 F1: 0.8889, Class 1 F1: 0.5192
Class 1 Precision: 0.6959, Class 1 Recall: 0.4141
Overall Macro F1: 0.7041
--- End compute_token_metrics ---

--- Inside compute_token_metrics ---
Number of non-ignored tokens: 123010
True label counts: {np.int64(0): np.int64(94063), np.int64(1): np.int64(28947)}
Predicted label counts: {np.int64(0): np.int64(89599), np.int64(1): 


--- Inside compute_token_metrics ---
Number of non-ignored tokens: 123010
True label counts: {np.int64(0): np.int64(94063), np.int64(1): np.int64(28947)}
Predicted label counts: {np.int64(0): np.int64(89599), np.int64(1): np.int64(33411)}
Class 0 F1: 0.8687, Class 1 F1: 0.6133
Class 1 Precision: 0.5724, Class 1 Recall: 0.6606
Overall Macro F1: 0.7410
--- End compute_token_metrics ---
Token classification evaluation results: {'eval_loss': 0.40445396304130554, 'eval_token_macro_f1': 0.7410224210739902, 'eval_token_f1_class0': 0.8687153575589942, 'eval_token_f1_class1': 0.6133294845889862, 'eval_token_precision_class1': 0.5723564095657119, 'eval_token_recall_class1': 0.6606211351780841, 'eval_runtime': 16.068, 'eval_samples_per_second': 47.61, 'eval_steps_per_second': 2.987, 'epoch': 4.0}
***** eval metrics *****
  epoch                       =        4.0
  eval_loss                   =     0.4045
  eval_runtime                = 0:00:16.06
  eval_samples_per_second     =      47.61
  eva

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Tokenizer also saved.
Best model saved to ./token+span_approach_testing


Copy to Google Drive

In [12]:
# Run this cell AFTER trainer.train() finishes

from google.colab import drive
import os
import shutil # Import shutil for potentially safer copying

# 1. Mount Google Drive (if not already mounted)
try:
    drive.mount('/content/drive', force_remount=True) # force_remount can be helpful
    drive_mounted = True
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    drive_mounted = False

if drive_mounted:
    # 2. Define the temporary output directory where Trainer saved the model
    #    (MUST match the output_dir in your TrainingArguments from Step 8)
    temp_output_dir = "./token+span_approach_testing" # Replace with your actual output_dir if different

    # 3. Define the destination path in your Google Drive
    #    (You can change 'unlp_models' to your preferred folder name)
    drive_destination_parent = "/content/drive/MyDrive/unlp_models/"
    # Create the parent directory in Drive if it doesn't exist
    os.makedirs(drive_destination_parent, exist_ok=True)
    # Define the specific destination folder name (can be same as temp output)
    model_folder_name = os.path.basename(temp_output_dir) # Extracts 'results-span-binary-roberta-large'
    drive_destination_path = os.path.join(drive_destination_parent, model_folder_name)

    # 4. Copy the entire model directory
    if os.path.exists(temp_output_dir):
        print(f"Copying contents from '{temp_output_dir}' to '{drive_destination_path}'...")
        try:
            # Use shutil.copytree for robust directory copying
            if os.path.exists(drive_destination_path):
                 print("Destination already exists, removing it first...")
                 shutil.rmtree(drive_destination_path) # Remove existing to avoid merging issues
            shutil.copytree(temp_output_dir, drive_destination_path)
            print("Model successfully copied to Google Drive!")
        except Exception as e:
            print(f"Error copying files using shutil: {e}")
            print("Attempting copy using !cp command as fallback...")
            # Fallback using shell command (might be less robust with many files)
            !cp -r "{temp_output_dir}" "{drive_destination_parent}"
            # Check if copy likely succeeded (basic check)
            if os.path.exists(drive_destination_path):
                 print("Model copied to Google Drive using !cp (please verify contents).")
            else:
                 print("Copying failed using !cp as well.")

    else:
        print(f"Error: Temporary output directory '{temp_output_dir}' not found. Was training completed and saved?")

Mounted at /content/drive
Copying contents from './token+span_approach_testing' to '/content/drive/MyDrive/unlp_models/token+span_approach_testing'...
Model successfully copied to Google Drive!


Load Model

In [13]:
# Step 11: Load Best Model

from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# --- Point to the location where the BEST model was saved ---
# This should match the directory you copied TO Google Drive
saved_model_path = "/content/drive/MyDrive/unlp_models/token+span_approach_testing"
# Alternatively, if still connected and want to use local copy:
# saved_model_path = "./results-span-binary-roberta-large"
# ---

try:
    print(f"Loading tokenizer from: {saved_model_path}")
    tokenizer = AutoTokenizer.from_pretrained(saved_model_path)

    print(f"Loading BEST fine-tuned model from: {saved_model_path}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForTokenClassification.from_pretrained(saved_model_path).to(device)
    model.eval() # Set model to evaluation mode

    print("\nBest fine-tuned model and tokenizer loaded successfully.")
    print(f"Model is on device: {device}")

except OSError:
    print(f"Error: Cannot find model or tokenizer files in '{saved_model_path}'.")
    print("Please ensure the path is correct and the model files exist (e.g., check Google Drive).")
    model = None # Prevent proceeding if loading fails
except Exception as e:
    print(f"An error occurred loading the model/tokenizer: {e}")
    model = None

Loading tokenizer from: /content/drive/MyDrive/unlp_models/token+span_approach_testing
Loading BEST fine-tuned model from: /content/drive/MyDrive/unlp_models/token+span_approach_testing

Best fine-tuned model and tokenizer loaded successfully.
Model is on device: cuda


Load Test Data

In [14]:
# Step 11.1: Load Test Data
# Make sure test.csv is uploaded to your Colab environment or accessible via path
test_df = pd.read_csv("test.csv")

# Optional: Convert to Hugging Face Dataset if needed for some processing later,
# but we'll mostly iterate through the DataFrame.
from datasets import Dataset
test_dataset = Dataset.from_pandas(test_df)

print("Test data loaded:")
print(test_df.head())
print("\nTest Dataset info:")
print(test_dataset)

Test data loaded:
                                     id  \
0  521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba   
1  9b2a61e4-d14e-4ff7-b304-e73d720319bf   
2  f0f1c236-80a8-4d25-b30c-a420a39be632   
3  31ea05ba-2c2b-4b84-aba7-f3cf6841b204   
4  a79e13ec-6d9a-40b5-b54c-7f4f743a7525   

                                             content  
0  Они просрали нашу технику, положили кучу людей...  
1  ❗️\nКитай предлагает отдать оккупированные тер...  
2  Сегодня будет ровно 6 месяцев с этого обещания...  
3  ⚡️\nІзраїль вперше у світі збив балістичну рак...  
4  Склав невелику навчально-методичну таблицю на ...  

Test Dataset info:
Dataset({
    features: ['id', 'content'],
    num_rows: 5735
})


Eval data ready

In [15]:
# Step 12: Verify Eval Split is Available

if 'eval_split' in locals() and eval_split is not None:
    print(f"Evaluation split ('eval_split') is available with {len(eval_split)} examples.")
    # Check first example's structure
    # print(eval_split[0])
else:
    print("Error: 'eval_split' not found. Please ensure Step 4 ran correctly.")
    # Stop if eval_split isn't available
    eval_split = None

Evaluation split ('eval_split') is available with 765 examples.


Span reconstruction

In [16]:
# Step 13: Define Span Reconstruction Function

import torch # Ensure torch is imported

def predict_spans(text, model_to_use, tokenizer_to_use):
    # Tokenize the input text and return offsets as tensors
    # Use the tokenizer passed as argument
    tokenized_inputs = tokenizer_to_use(
        text,
        truncation=True,
        padding="max_length", # Use same padding as training
        max_length=512,      # Use same max_length as training
        return_offsets_mapping=True,
        return_tensors="pt"
    )

    # Extract offset mapping and convert it to a list (on CPU)
    offset_mapping = tokenized_inputs.pop("offset_mapping")[0].tolist()

    # Move inputs to the same device as the model
    device = next(model_to_use.parameters()).device
    tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}

    # Forward pass using the passed model
    model_to_use.eval() # Ensure eval mode
    with torch.no_grad():
        outputs = model_to_use(**tokenized_inputs)
    logits = outputs.logits # shape: (1, seq_length, 2)

    # Get binary predictions (0 or 1)
    preds = torch.argmax(logits, dim=-1).squeeze().cpu().numpy() # shape: (seq_length,)

    # Convert token-level predictions (0/1) to character-level spans
    spans = []
    current_span = None
    for i, (token_start, token_end) in enumerate(offset_mapping):
        # Skip special tokens or padding
        if token_start == token_end:
             # If we were in a span, close it before skipping
             if current_span is not None:
                  spans.append(tuple(current_span))
                  current_span = None
             continue

        # Check the prediction for the current token index
        token_prediction = preds[i]

        if token_prediction == 1:
            # If token is predicted as manipulative (1)
            if current_span is None:
                # Start a new span
                current_span = [token_start, token_end]
            else:
                # Extend the current span to this token's end
                current_span[1] = token_end
        else:
            # If token is predicted as non-manipulative (0)
            if current_span is not None:
                # Close the current span if we were in one
                spans.append(tuple(current_span))
                current_span = None
            # Otherwise, just continue (token is O and no span is active)

    # After the loop, check if a span was still open
    if current_span is not None:
        spans.append(tuple(current_span))

    # Return list of (start_char, end_char) tuples
    return spans

print("predict_spans function defined.")

predict_spans function defined.


Span predictions on Test Data

In [17]:
# Step 14: Generate Predictions for the Test Set

import pandas as pd
from tqdm.auto import tqdm # For progress bar

test_predictions = []

print(f"Generating predictions for {len(test_df)} test samples...")

# Ensure model is on the correct device (CPU or GPU)
# model.to(device) # Already done in Step 11, but good practice to ensure

# Iterate through the test DataFrame
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    text_id = row['id']
    text_content = row['content']

    # Handle potential NaN/empty content
    if pd.isna(text_content):
        predicted_spans_list = [] # Predict empty list for empty content
    else:
        # Use the function defined in Step 13
        predicted_spans_list = predict_spans(text_content, model, tokenizer)

    # Store the id and the list of predicted spans
    # Convert the list of tuples directly to the required string format for submission
    trigger_words_str = str(predicted_spans_list)
    test_predictions.append({'id': text_id, 'trigger_words': trigger_words_str})

# Convert the list of dictionaries into a DataFrame matching the submission format
submission_df = pd.DataFrame(test_predictions)

print("\nSample predictions:")
print(submission_df.head())

# Optional: Save the submission file
# submission_df.to_csv("submission.csv", index=False)
# print("\nSubmission DataFrame created and optionally saved to submission.csv")

Generating predictions for 5735 test samples...


  0%|          | 0/5735 [00:00<?, ?it/s]


Sample predictions:
                                     id  \
0  521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba   
1  9b2a61e4-d14e-4ff7-b304-e73d720319bf   
2  f0f1c236-80a8-4d25-b30c-a420a39be632   
3  31ea05ba-2c2b-4b84-aba7-f3cf6841b204   
4  a79e13ec-6d9a-40b5-b54c-7f4f743a7525   

                                       trigger_words  
0                                         [(0, 253)]  
1   [(324, 327), (339, 342), (346, 349), (374, 425)]  
2                  [(32, 75), (76, 127), (142, 143)]  
3                                                 []  
4  [(55, 59), (67, 72), (76, 81), (87, 103), (127...  


Eval spans

In [18]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.0 MB/s eta 0:00:00


In [19]:
!pip install seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [20]:
# Step 15: Evaluate Test Predictions using Official Metric

# --- Ensure these function definitions are present from your original Step 15 ---
import pandas as pd
import ast

class ParticipantVisibleError(Exception):
    """Custom exception for participant-visible errors."""
    pass

def safe_parse_spans(trigger_words):
    # (Keep the function definition as you have it)
    if isinstance(trigger_words, str):
        try:
            # Make sure it handles empty lists "[]" correctly
            parsed = ast.literal_eval(trigger_words)
            if isinstance(parsed, list):
                # Validate structure: list of (int, int) tuples/lists
                return [(int(s), int(e)) for s, e in parsed if isinstance(s, int) and isinstance(e, int) and s <= e]
            else:
                return [] # Not a list after eval
        except (ValueError, SyntaxError):
            return [] # Parsing error or empty string
    elif isinstance(trigger_words, list): # Handle cases where it might already be a list
         # Validate structure: list of (int, int) tuples/lists
         return [(int(s), int(e)) for s, e in trigger_words if isinstance(s, (int, float)) and isinstance(e, (int, float)) and int(s) <= int(e)] # Allow float conversion just in case
    return [] # Default to empty list for other types or issues


def extract_tokens_from_spans(spans):
    # (Keep the function definition as you have it)
    tokens = set()
    for start, end in spans:
        # Ensure start and end are integers before creating range
        try:
            start_int = int(start)
            end_int = int(end)
            if start_int < end_int: # Spans should have length > 0
                 tokens.update(range(start_int, end_int))
        except (ValueError, TypeError):
            continue # Skip invalid span components
    return tokens

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    # (Keep the function definition as you have it,
    # ensuring it uses safe_parse_spans and extract_tokens_from_spans correctly)

    # Example check (ensure this is in your function definition)
    if not all(col in solution.columns for col in [row_id_column_name, "trigger_words"]):
        raise ValueError(f"Solution DataFrame must contain '{row_id_column_name}' and 'trigger_words' columns.")
    if not all(col in submission.columns for col in [row_id_column_name, "trigger_words"]):
        raise ValueError(f"Submission DataFrame must contain '{row_id_column_name}' and 'trigger_words' columns.")

    solution = solution.copy()
    submission = submission.copy()

    solution["trigger_words"] = solution["trigger_words"].apply(safe_parse_spans)
    submission["trigger_words"] = submission["trigger_words"].apply(safe_parse_spans)

    merged = pd.merge(
        solution[[row_id_column_name, "trigger_words"]],  # Select only necessary columns
        submission[[row_id_column_name, "trigger_words"]],
        on=row_id_column_name,
        suffixes=("_solution", "_submission")
    )

    # Ensure merged is not empty if submission/solution had non-matching IDs
    if merged.empty and (not solution.empty and not submission.empty):
         print(f"Warning: No matching IDs found between solution and submission using column '{row_id_column_name}'.")
         # Depending on competition rules, this might be 0 or raise an error. Let's return 0.
         return 0.0
    elif merged.empty:
         # Handle case where either solution or submission is empty
         return 0.0


    total_true_chars = 0
    total_pred_chars = 0
    overlapping_chars = 0

    for _, row in merged.iterrows():
        true_spans = row["trigger_words_solution"]
        pred_spans = row["trigger_words_submission"]

        true_chars = extract_tokens_from_spans(true_spans)
        pred_chars = extract_tokens_from_spans(pred_spans)

        total_true_chars += len(true_chars)
        total_pred_chars += len(pred_chars)
        overlapping_chars += len(true_chars.intersection(pred_chars))

    precision = overlapping_chars / total_pred_chars if total_pred_chars > 0 else 0
    recall = overlapping_chars / total_true_chars if total_true_chars > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print("\n--- Official Character-Overlap Metrics ---")
    print(f"Total True Characters in Spans:      {total_true_chars}")
    print(f"Total Predicted Characters in Spans: {total_pred_chars}")
    print(f"Overlapping Characters:            {overlapping_chars}")
    print(f"Precision:                         {precision:.4f}")
    print(f"Recall:                            {recall:.4f}")
    print(f"F1-Score:                          {f1:.4f}") # This is the key metric

    return f1

# --- End of function definitions ---


# --- Add this part to load the solution and call the score function ---
print("\nLoading solution file...")
# Make sure solution.csv is uploaded or accessible via path
solution_df = pd.read_csv("solution.csv")

print("Evaluating submission against solution...")

# Ensure the submission_df from Step 14 is available
if 'submission_df' in locals():
    # Call the score function with the correct dataframes and ID column name
    final_f1_score = score(solution=solution_df, submission=submission_df, row_id_column_name="id")
    print(f"\nFinal Official F1 Score on Test Set: {final_f1_score:.4f}")
else:
    print("Error: submission_df not found. Please ensure Step 14 ran successfully.")


Loading solution file...
Evaluating submission against solution...

--- Official Character-Overlap Metrics ---
Total True Characters in Spans:      821072
Total Predicted Characters in Spans: 870302
Overlapping Characters:            501339
Precision:                         0.5761
Recall:                            0.6106
F1-Score:                          0.5928

Final Official F1 Score on Test Set: 0.5928
